# 06 — Monitoring: data / model / bias monitors + CloudWatch dashboard

**Team Update 5.1 (all five bullets):** model monitors, data monitors,
infrastructure monitors, a CloudWatch monitoring dashboard, and SageMaker
model/data reports.

Architecture (design doc, batch system with a monitoring endpoint): the
champion is deployed to a small **real-time endpoint used only for
monitoring**; the reserved production cohort (labels already known) is
replayed through it with data capture on. That gives the Clarify monitors a
live channel exactly as in Lab 5.1, while the batch transform remains the
production scorer.

| Step | Verify (📸) |
|---|---|
| Monitoring endpoint + capture + replay | invoke loop output |
| Data-quality monitor (baseline + schedule + execution) | constraints violations |
| Model-quality monitor (production ground truth) | quality report metrics |
| Model-bias monitor (facet: gender one-hot) | bias report metrics |
| CloudWatch dashboard + infra alarms | dashboard body / alarms |
| Reports in S3 | monitoring/reports listing |

**Cost guard:** the endpoint is deleted and schedules stopped in the cleanup
cell - run it after screenshots (~$0.25/hr while the endpoint lives).

## 1. Setup

In [ ]:
import os
from pathlib import Path

# CWD depends on the runner: Studio file view starts at the repo root, while
# `jupyter nbconvert` starts in the notebook's directory. Resolve the repo
# root (the dir that contains src/) and work from there.
if (Path.cwd() / "src").is_dir():
    REPO = Path.cwd()
elif (Path.cwd().parent / "src").is_dir():
    REPO = Path.cwd().parent
else:  # fresh Studio file view: clone the repo next to the notebooks dir
    !git clone --quiet https://github.com/University-San-Diego-MAAI/AAI-540-Group-1-Final-Project.git ../repo
    REPO = Path.cwd().parent / "repo"
os.chdir(REPO)
print("repo root:", REPO, "| src/model_common.py:", (REPO / "src" / "model_common.py").is_file())

In [ ]:
%pip install -q "sagemaker<3"

import json
import time

import boto3
import pandas as pd
import sagemaker

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sm_client = boto_session.client("sagemaker", region_name=region)
runtime = boto_session.client("sagemaker-runtime", region_name=region)
s3 = boto3.client("s3", region_name=region)
cw = boto3.client("cloudwatch", region_name=region)
bucket = sagemaker.Session(boto_session=boto_session).default_bucket()
prefix = "aai-540-g1"
MON = f"{prefix}/monitoring"

try:
    role = sagemaker.get_execution_role()
except Exception:
    iam = boto3.client("iam", region_name=region)
    by_name = {r["RoleName"]: r for r in iam.list_roles()["Roles"]}
    if "aai-540-g1-execution-role" in by_name:
        picked = "aai-540-g1-execution-role"
    else:
        cands = [n for n in by_name if n.startswith("AmazonSageMaker-ExecutionRole")]
        picked = sorted(cands, key=lambda n: by_name[n]["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{picked}"
print(f"role={role}\nbucket={bucket}")

## 2. Production cohort + ground truth (from notebook 05's scoring)

Loads the scored production cohort and joins the reserved labels (the
40% production slice was scored blind; its labels feed the quality monitor).

In [ ]:
s3.download_file(bucket, f"{prefix}/artifacts/scoring/production_scores.csv",
                 "production_scores.csv")
s3.download_file(bucket, f"{prefix}/processed/training/training_table.csv",
                 "training_table.csv")
scores = pd.read_csv("production_scores.csv")
table = pd.read_csv("training_table.csv", dtype={"person_id": int})

prod = table[table["split"] == "production"][["person_id", "risk_class"]]
ground = prod.merge(scores[["person_id", "predicted_risk"]], on="person_id")
print(f"{len(ground):,} production members with ground truth; "
      f"agreement: {(ground['risk_class'] == ground['predicted_risk']).mean():.3f}")

## 3. Monitoring endpoint with data capture

In [ ]:
ENDPOINT = "member-utilization-risk-monitor"
CAPTURE = f"s3://{bucket}/{MON}/capture"

try:
    sm_client.create_endpoint_config(
        EndpointConfigName=ENDPOINT,
        ProductionVariants=[{
            "VariantName": "monitor", "ModelName": "member-utilization-risk-champion",
            "InstanceType": "ml.m5.xlarge", "InitialInstanceCount": 1,
            "InitialVariantWeight": 1,
        }],
        DataCaptureConfig={
            "EnableCapture": True, "InitialSamplingPercentage": 100,
            "DestinationS3Uri": CAPTURE,
            "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
            "CaptureContentTypeHeader": {"CsvContentTypes": ["text/csv"]},
        },
    )
    sm_client.create_endpoint(EndpointName=ENDPOINT, EndpointConfigName=ENDPOINT)
    print("endpoint creating...")
except sm_client.exceptions.ResourceInUse:
    print("endpoint/config already exist")
except sm_client.exceptions.ClientError as e:
    if "exist" in str(e).lower():  # "already existing ..." variants
        print("endpoint/config already exist")
    else:
        raise

sm_client.get_waiter("endpoint_in_service").wait(EndpointName=ENDPOINT, WaiterConfig={"Delay": 30, "MaxAttempts": 20})
print(f"{ENDPOINT} in service")

## 4. Replay the production cohort through the endpoint

Body format matches notebook 05's scoring input: the encoded features in the
exact training column order (no id, no header). 1,500 rows keeps the replay
under ~10 minutes; the rest stay available for a second run.

In [ ]:
import sys
sys.path.insert(0, "src")
from model_common import encode_features, load_table

X_all, feature_names, audit = encode_features(load_table("training_table.csv"))
prod_mask = (audit["split"] == "production").to_numpy()
prod_X = pd.DataFrame(X_all[prod_mask], columns=feature_names).head(1500)

REPLAYED = []
for i, row in prod_X.iterrows():
    body = ",".join(f"{v:.6g}" for v in row.to_numpy())
    resp = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv", Body=body)
    REPLAYED.append(resp["Body"].read().decode().strip())
    if len(REPLAYED) % 300 == 0:
        print(f"{len(REPLAYED)} invoked")
print(f"replayed {len(REPLAYED)} rows; capture -> {CAPTURE}")

## 5. Data-quality monitor (baseline + schedule + first execution)

Baseline = the training split (the distribution production traffic should
resemble). Schedule runs hourly; we trigger one execution immediately and
read the constraint-violations report.

In [ ]:
from sagemaker.model_monitor import CronExpressionGenerator, DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

train_rows = pd.DataFrame(X_all[split := (audit["split"] == "train").to_numpy()],
                          columns=feature_names)
train_rows.to_csv("baseline_train.csv", index=False, header=False)
s3.upload_file("baseline_train.csv", bucket, f"{MON}/baseline/baseline_train.csv")

data_monitor = DefaultModelMonitor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, max_runtime_in_seconds=1800,
)
data_monitor.suggest_baseline(
    baseline_dataset=f"s3://{bucket}/{MON}/baseline/baseline_train.csv",
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=f"s3://{bucket}/{MON}/data-baseline",
    wait=True,
)
print("data baseline suggested")

In [ ]:
from sagemaker.model_monitor import CronExpressionGenerator

data_monitor.create_monitoring_schedule(
    monitor_schedule_name="aai540-g1-data-quality",
    endpoint_input=ENDPOINT,
    output_s3_uri=f"s3://{bucket}/{MON}/data-quality",
    statistics=data_monitor.baseline_statistics(),
    constraints=data_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)
print("data-quality schedule created (hourly)")

""## 6. Model-quality monitor (production ground truth)

Ground truth merge file: one JSON object per captured event. The capture uses
sequential auto-increment event ids, so we replay them in the same order.

In [ ]:
import re

objs = s3.list_objects_v2(Bucket=bucket, Prefix=f"{MON}/capture/")
keys = sorted(o["Key"] for o in objs.get("Contents", []) if o["Key"].endswith(".jsonl.gz"))
print(f"capture files: {len(keys)}")

# map capture event order -> ground-truth labels of the replayed rows
gt_rows = []
for i, line in enumerate(REPLAYED):
    true_label = prod.iloc[i]["risk_class"] if i < len(prod) else None
    gt_rows.append({"eventId": str(100000 + i),
                    "groundTruthData": {"data": true_label}})
pd.DataFrame(gt_rows).to_json("ground_truth.jsonl", orient="records", lines=True)
s3.upload_file("ground_truth.jsonl", bucket, f"{MON}/ground_truth/ground_truth.jsonl")
print(f"ground truth rows: {len(gt_rows)}")

In [ ]:
from sagemaker.model_monitor import ModelQualityMonitor

quality_monitor = ModelQualityMonitor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, max_runtime_in_seconds=1800,
)
quality_monitor.create_monitoring_schedule(
    monitor_schedule_name="aai540-g1-model-quality",
    endpoint_input=sagemaker.model_monitor.EndpointInput(
        endpoint_name=ENDPOINT,
        destination=f"s3://{bucket}/{MON}/quality/input",
        start_time_offset="-PT1H", end_time_offset="-PT0H",
    ),
    ground_truth_input=f"s3://{bucket}/{MON}/ground_truth",
    problem_type="MulticlassClassification",
    inference_attribute="0",  # predicted label column in the capture
    probability_attribute="0",
    probability_threshold_attribute="0.5",
    output_s3_uri=f"s3://{bucket}/{MON}/quality",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)
print("model-quality schedule created (hourly)")

## 7. Model-bias monitor (facet: gender one-hot)

`race` stays an audit slice offline; the live bias monitor uses `gender_Male`
as the facet column so the Update-5.1 bias report is reproducible from
captured traffic. The schedule evaluates the captured production traffic
against the champion every hour.

In [ ]:
from sagemaker.clarify import BiasConfig, ModelConfig
from sagemaker.model_monitor import ModelBiasMonitor

bias_config = BiasConfig(
    label_values_or_threshold=["Low", "Medium", "High"],
    facet_name="gender_Male", facet_values_or_threshold=[1],
)
model_config = ModelConfig(
    model_name="member-utilization-risk-champion",
    instance_type="ml.m5.xlarge", instance_count=1,
    content_type="text/csv", accept_type="text/csv",
)
bias_monitor = ModelBiasMonitor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, max_runtime_in_seconds=1800,
)
bias_monitor.create_monitoring_schedule(
    monitor_schedule_name="aai540-g1-model-bias",
    endpoint_input=ENDPOINT,
    bias_config=bias_config,
    model_config=model_config,
    output_s3_uri=f"s3://{bucket}/{MON}/bias",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)
print("model-bias schedule created (hourly)")

## 8. Infrastructure monitors + 📊 CloudWatch dashboard

In [ ]:
for alarm_name, metric, threshold in [
    ("aai540-g1-endpoint-5xx", ("AWS/SageMaker", "ModelInvocation5XXErrors", "EndpointName", ENDPOINT), 1),
    ("aai540-g1-endpoint-latency", ("AWS/SageMaker", "ModelLatency", "EndpointName", ENDPOINT), 30000),
]:
    namespace, mname, dim, dval = metric
    cw.put_metric_alarm(
        AlarmName=alarm_name, Namespace=namespace, MetricName=mname,
        Dimensions=[{"Name": dim, "Value": dval}],
        Statistic="Sum" if "5XX" in mname else "Average",
        Period=300, EvaluationPeriods=1, Threshold=threshold,
        ComparisonOperator="GreaterThanOrEqualToThreshold",
    )
print("infra alarms created: endpoint 5XX + latency")

In [ ]:
dashboard = {
    "widgets": [
        {"type": "metric", "x": 0, "y": 0, "width": 12, "height": 6,
         "properties": {"title": "Endpoint invocations + 5XX",
                        "metrics": [["AWS/SageMaker", "Invocations", "EndpointName", ENDPOINT],
                                    [".", "ModelInvocation5XXErrors", ".", "."]],
                        "period": 300, "stat": "Sum", "region": region}},
        {"type": "metric", "x": 12, "y": 0, "width": 12, "height": 6,
         "properties": {"title": "Endpoint latency (ms)", "metrics": [
             ["AWS/SageMaker", "ModelLatency", "EndpointName", ENDPOINT]],
                        "period": 300, "stat": "Average", "region": region}},
        {"type": "metric", "x": 0, "y": 6, "width": 12, "height": 6,
         "properties": {"title": "Data-quality violations",
                        "metrics": [["AWS/SageMaker/ModelMonitoring", "FeaturesViolated",
                                     "MonitoringSchedule", "aai540-g1-data-quality"]],
                        "period": 3600, "stat": "Average", "region": region}},
        {"type": "metric", "x": 12, "y": 6, "width": 12, "height": 6,
         "properties": {"title": "Batch transform duration",
                        "metrics": [["AWS/SageMaker", "TransformJobDuration", "Host", "Algorithm"]],
                        "period": 3600, "stat": "Average", "region": region}},
    ]
}
cw.put_dashboard(DashboardName="aai540-g1-monitoring", DashboardBody=json.dumps(dashboard))
print("dashboard aai540-g1-monitoring published")

## 9. 📸 First monitoring executions + reports (Update 5.1: SageMaker reports)

Schedules run hourly on their own; to collect report evidence now, start
executions manually and read back the metrics/violations JSON.

In [ ]:
def start_and_wait(schedule):
    name = schedule if isinstance(schedule, str) else schedule.monitoring_schedule_name
    sm_client.start_monitoring_schedule(MonitoringScheduleName=name)
    print(f"started {name}")
    for _ in range(60):
        desc = sm_client.describe_monitoring_schedule(MonitoringScheduleName=name)
        jobs = desc.get("LastMonitoringExecutionSummary")
        status = jobs["MonitoringExecutionStatus"] if jobs else "Pending"
        if status == "Completed":
            return jobs["MonitoringExecutionArn"]
        if status == "Failed":
            raise RuntimeError(desc)
        time.sleep(20)
    raise TimeoutError(name)

exec_arn = start_and_wait("aai540-g1-data-quality")
print("data-quality execution:", exec_arn)

In [ ]:
# report artifacts land under monitoring/<schedule>/: list + show violations
objs = s3.list_objects_v2(Bucket=bucket, Prefix=f"{MON}/data-quality/")
for o in objs.get("Contents", [])[:12]:
    print(o["Key"], o["Size"])

## 10. Cleanup (run after screenshots — stops recurring charges)

```python
for name in ["aai540-g1-data-quality", "aai540-g1-model-quality", "aai540-g1-model-bias"]:
    try:
        sm_client.stop_monitoring_schedule(MonitoringScheduleName=name)
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=name)
    except Exception as e:
        print(name, e)
sm_client.delete_endpoint(EndpointName=ENDPOINT)
sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT)
# keep the registered model + scores for the final report
print("cleanup done")
```